<a href="https://colab.research.google.com/github/Saadd-x/FYP-Work/blob/main/Drone%20Dataset%20Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

FYP_DIR = "/content/drive/MyDrive/FYP_Drone"
os.makedirs(FYP_DIR, exist_ok=True)

print("FYP folder:", FYP_DIR)

Mounted at /content/drive
FYP folder: /content/drive/MyDrive/FYP_Drone


In [ ]:
!pip install -q huggingface_hub

from huggingface_hub import snapshot_download

REPO_ID = "lgrzybowski/seraphim-drone-detection-dataset"
DATASET_RAW = "/content/seraphim"

snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    local_dir=DATASET_RAW
)

print("Dataset downloaded.")

In [ ]:
import zipfile
from pathlib import Path

dataset_path = Path(DATASET_RAW)

for zip_path in dataset_path.rglob("*.zip"):
    print("Extracting:", zip_path)

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(zip_path.parent)

print("Extraction complete.")

In [ ]:
import random
import shutil
from pathlib import Path

random.seed(42)

SOURCE = Path("/content/seraphim")
BASE = Path("/content/drone_dataset")

for folder in [
    BASE / "images/train",
    BASE / "images/val",
    BASE / "labels/train",
    BASE / "labels/val",
]:
    folder.mkdir(parents=True, exist_ok=True)

source_images = SOURCE / "train/images"
source_labels = SOURCE / "train/labels"

images = [
    img for img in source_images.glob("*")
    if (source_labels / f"{img.stem}.txt").exists()
]

random.shuffle(images)

split = int(len(images) * 0.90)

train_images = images[:split]
val_images = images[split:]

print("Total:", len(images))
print("Train:", len(train_images))
print("Validation:", len(val_images))

for img in train_images:
    shutil.copy2(img, BASE / "images/train" / img.name)
    shutil.copy2(
        source_labels / f"{img.stem}.txt",
        BASE / "labels/train" / f"{img.stem}.txt"
    )

for img in val_images:
    shutil.copy2(img, BASE / "images/val" / img.name)
    shutil.copy2(
        source_labels / f"{img.stem}.txt",
        BASE / "labels/val" / f"{img.stem}.txt"
    )

print("Dataset prepared.")

In [ ]:
data_yaml = """
path: /content/drone_dataset

train: images/train
val: images/val

nc: 1
names:
  0: drone
"""

with open("/content/drone_dataset/data.yaml", "w") as f:
    f.write(data_yaml.strip())

print(data_yaml)

In [ ]:
%cd /content

!git clone https://github.com/ultralytics/yolov5.git
%cd /content/yolov5

!pip install -q -r requirements.txt

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1),
          "GB")

In [ ]:
%cd /content/yolov5

!python train.py \
    --img 640 \
    --batch 16 \
    --epochs 20 \
    --data /content/drone_dataset/data.yaml \
    --weights yolov5s.pt \
    --name test_drone \
    --project /content/drive/MyDrive/FYP_Drone/runs \
    --save-period 1

/content/yolov5
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
WARNING ⚠️ wandb: wandb is deprecated and will be removed in a future release. See supported integrations at https://github.com/ultralytics/yolov5#integrations.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice: (30 second timeout) 
wandb: WARNING W&B disabled due to login timeout.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
train: weights=yolov5s.pt, cfg=, data=/content/drone_dataset/data.yaml, hyp=data/hyps/hyp.scratch-low.yaml, epochs=20, batch_size=16, imgsz=640, rect=False, resume=False, nosave=False, noval=False, noautoanchor=False, noplots=False, evolve=None,

In [ ]:
%cd /content/yolov5

!python train.py \
    --resume /content/drive/MyDrive/FYP_Drone/runs/test_drone/weights/last.pt

In [ ]:
!python detect.py \
    --weights /content/drive/MyDrive/FYP_Drone/runs/test_drone/weights/best.pt \
    --source "/content/your_video.mp4" \
    --img 640 \
    --conf 0.25

In [ ]:
Google Drive/FYP_Drone/videos/